# 05 - Pretrained baselines (frozen backbones)

**What this notebook does**: trains four ImageNet-pretrained backbones with a small classification
head on the `faithful` split, so MiniConvNet can be compared against standard architectures on
identical data: `ResNet50`, `VGG16`, `MobileNetV3Small`, `EfficientNetV2B0`.

> **[CHOICE] Frozen backbones, four models, CPU scope.** The backbone is frozen
> (`trainable_base=False`) and only the head is trained. Full fine-tuning of a 23M-parameter backbone
> is not affordable on CPU, and feature extraction is the standard comparison protocol the paper's
> table uses anyway. v2's optional extended set (VGG19, InceptionV3, ConvNeXtTiny) is out of scope
> this round. State both decisions whenever these numbers are quoted.

**What must already exist**: split CSVs from notebook 00, and internet access the first time (Keras
downloads the ImageNet weights; they are cached afterwards).

**CPU cost warning**: these backbones are far heavier per epoch than MiniConvNet even with the
backbone frozen - every epoch still runs a full forward pass through ResNet50/VGG16. Each model gets
its own time estimate **and its own cell**, so a slow one can be skipped without losing the others.

**Where results go**: `outputs/results_table.csv` as canonical rows (one per model), since a single
feature-extraction run is the standard protocol. Every run is collapse-checked and saves raw
predictions.

**What "looks right"**: baselines clearly above chance within a few epochs (frozen ImageNet features
converge fast), and the familiar pattern from both previous attempts - high tumour-vs-healthy
accuracy with much weaker subtype discrimination (LESSON 11).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('baseline epochs:', EPOCHS_BASELINE, '| lr:', LR_BASELINE)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_baseline, count_params, DEFAULT_BASELINES
from src.train_utils import (set_global_seeds, compute_report, compile_model, optimizer_summary,
                             class_weights_for, make_callbacks, make_epoch_timer, save_history,
                             plot_history, final_epoch_summary, run_name_for,
                             estimate_training_time)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                                plot_confusion_matrix, tumor_vs_subtype_breakdown,
                                interpret_breakdown, save_predictions, result_row_from_metrics,
                                record_canonical, load_results)

set_global_seeds(SEED)
for k, v in compute_report().items():
    print(f'{k}: {v}')
print()
print('baselines to train:', DEFAULT_BASELINES)

In [ ]:
SPLIT_FOR_BASELINES = 'faithful'

sdf = load_split(SPLIT_FOR_BASELINES)
# Baselines use sparse labels: label smoothing is a MiniConvNet anti-collapse
# measure (LESSON 2), not part of the baseline comparison protocol.
train_ds, val_ds, test_ds, frames = make_split_datasets(sdf)
class_weight = class_weights_for(SPLIT_FOR_BASELINES, frames['train']['label'].values)

print(split_counts(sdf).to_string())
print('class_weight:', class_weight if class_weight else 'None (by design for faithful)')

## 1. Baseline runner

In [ ]:
def run_baseline(name, epochs=EPOCHS_BASELINE, trainable_base=False):
    set_global_seeds(SEED)
    run_name = run_name_for(name.lower(), SPLIT_FOR_BASELINES)
    print('=' * 72)
    print('baseline:', name, '| run:', run_name)

    model = build_baseline(name, trainable_base=trainable_base)
    compile_model(model, lr=LR_BASELINE, label_smoothing=0.0)   # sparse loss
    params = count_params(model)
    print('params   :', params)
    print('optimizer:', optimizer_summary(model))

    timer = make_epoch_timer(verbose=0)
    history = model.fit(train_ds, validation_data=val_ds, epochs=epochs,
                        class_weight=class_weight,
                        callbacks=make_callbacks(run_name, timer=timer), verbose=2)
    save_history(history, run_name, timer=timer)
    summary = final_epoch_summary(history, timer=timer)
    print('\n', summary)

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)
    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'split_variant': SPLIT_FOR_BASELINES, 'baseline': name,
                           'protocol': 'frozen backbone, head-only training'})

    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    breakdown = tumor_vs_subtype_breakdown(y_true, y_pred)

    print('\ntest metrics:', {k: round(v, 4) for k, v in metrics.items()})
    print()
    print_collapse_report(collapse, run_name)
    print('\n' + interpret_breakdown(breakdown))

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)

    record_canonical(result_row_from_metrics(
        model_name=name, metrics=metrics, collapse=collapse,
        arch_variant='transfer_frozen' if not trainable_base else 'transfer_finetuned',
        split_variant=SPLIT_FOR_BASELINES, params=params['total_params'],
        epochs_trained=summary['epochs_trained'], n_runs=1, breakdown=breakdown,
        notes=(f"ImageNet feature extraction, head-only training; "
               f"{summary.get('mean_seconds_per_epoch')}s/epoch, "
               f"{summary.get('total_minutes')} min total on CPU")))

    tf.keras.backend.clear_session()
    return {'model': name, 'run_name': run_name, 'status': collapse['status'],
            'params': params['total_params'],
            'binary_tumor_acc': breakdown['binary_tumor_vs_healthy_accuracy'],
            'subtype_acc': breakdown['subtype_accuracy_all_tumors'],
            'minutes': summary.get('total_minutes'), **metrics}

## 2. Time estimate for the whole baseline set

ResNet50 is timed as the representative model and the estimate is scaled by the number of baselines.
That is rough - VGG16 is slower per epoch, MobileNetV3Small much faster - so treat it as an
order-of-magnitude figure and re-check per model if it lands near the threshold.

**Read this before section 3.** If it exceeds the threshold, agree the cost first, or run a subset.

In [ ]:
est = estimate_training_time(
    model_fn=lambda: compile_model(build_baseline('ResNet50'), lr=LR_BASELINE,
                                   label_smoothing=0.0, verbose=False),
    train_ds=train_ds, val_ds=val_ds,
    planned_epochs=EPOCHS_BASELINE, n_runs=len(DEFAULT_BASELINES),
    class_weight=class_weight)
print()
print('Scaled from ResNet50 alone: VGG16 is typically slower per epoch and MobileNetV3Small much '
      'faster, so the per-model spread around this figure is wide.')

## 3. The four baselines, one cell each

Each architecture gets its own cell so a failed download, an OOM or a time overrun costs you that one
model rather than the whole notebook.

In [ ]:
results = {}
results['ResNet50'] = run_baseline('ResNet50')

In [ ]:
results['VGG16'] = run_baseline('VGG16')

In [ ]:
results['MobileNetV3Small'] = run_baseline('MobileNetV3Small')

In [ ]:
results['EfficientNetV2B0'] = run_baseline('EfficientNetV2B0')

## 4. Baseline summary

**Looks right**: four rows, all `ok`, all with far more parameters than MiniConvNet's ~499K - that
parameter gap is the entire point of the comparison. Note the `binary_tumor_acc` vs `subtype_acc`
columns: the pattern that held across both previous attempts is strong detection with much weaker
subtype discrimination (LESSON 11).

In [ ]:
tbl = pd.DataFrame(results.values())
print(tbl[['model', 'params', 'accuracy', 'f1_macro', 'cohen_kappa', 'binary_tumor_acc',
           'subtype_acc', 'minutes', 'status']].round(4).to_string(index=False))

bad = tbl[tbl['status'] != VALID_TAG]
print('\ninvalid baselines:', bad['model'].tolist() if len(bad) else 'none')
print('total baseline wall clock:', round(tbl['minutes'].fillna(0).sum(), 1), 'min')

In [ ]:
canon = load_results('canonical')
print(canon[['model', 'arch_variant', 'params', 'accuracy', 'accuracy_std', 'f1_macro',
             'binary_tumor_acc', 'subtype_acc', 'n_runs', 'status']].round(4).to_string(index=False))
print('\nnext: 06_evaluate_and_compare.ipynb')